In [6]:

%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

from verimon.utils import setup_logging

setup_logging()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from verimon import loaders
from random import randrange
from math import sqrt

gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 6**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [8]:

from stormvogel.mapping import stormpy_to_stormvogel

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
# show(mc_sv)

In [9]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in mc_sv.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

<IPython.core.display.Javascript object>

In [13]:
from aalpy import run_Lstar
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


false_pos_threshold = 0.6
false_neg_threshold = 0.6
horizon = 8
relative_error = 0.2

alphabet = ["init", "normal", "snake", "ladder"]

filtering_sul = FilteringSUL(
    mc, 
    "init", 
    alphabet, 
    'P>0.9 [ "good" ]', 
    false_neg_threshold, 
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc_sv,
    gb,
    false_pos_threshold,
    false_neg_threshold,
    horizon,
    'P>0.9 [ "good" ]',
    'good',
    relative_error
    
)
learned_monitor = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
)

Hypothesis 1: 1 states.
Visualization started in the background thread.
DEBUG:2024-10-18 11:05:16,849 - (0.00s) - MonitorLearning.py - Finding false negative probability 
DEBUG:2024-10-18 11:05:16,850 - (0.00s) - verify.py - Building model 
DEBUG:2024-10-18 11:05:16,851 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-18 11:05:16,856 - (0.00s) - verify.py - Pruning done 
INFO:2024-10-18 11:05:16,862 - (0.01s) - MDP_product.py - New good states become: [['[pos=9]']] 
DEBUG:2024-10-18 11:05:16,863 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-18 11:05:16,963 - (0.10s) - verify.py - creating product done 
Model saved to model1.pdf.
DEBUG:2024-10-18 11:05:16,964 - (0.00s) - verify.py - Creating trace 
INFO:2024-10-18 11:05:17,239 - (0.28s) - MDP_product.py - --------------------
Synthesis summary:
constraint 1: P>3/5 [F "goal"]

method: AR, synthesis time: 0.24 s
number of holes: 8, family size: 390625, quotient: 51 states / 247 actions
explored: 100 %
MDP stats: avg MDP size

In [17]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import *

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
result_goal, trace, assignment, product = false_negative(mc_sv, mon_cycl, gb, 8)

DEBUG:2024-10-18 11:07:47,730 - (71.60s) - verify.py - Building model 
DEBUG:2024-10-18 11:07:47,731 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-18 11:07:47,740 - (0.01s) - verify.py - Pruning done 
INFO:2024-10-18 11:07:47,746 - (0.01s) - MDP_product.py - New good states become: [['[pos=9]']] 
DEBUG:2024-10-18 11:07:47,747 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-18 11:07:50,692 - (2.95s) - verify.py - creating product done 
DEBUG:2024-10-18 11:07:50,693 - (0.00s) - verify.py - Creating trace 
INFO:2024-10-18 11:07:50,756 - (0.06s) - MDP_product.py - --------------------
Synthesis summary:
optimality objective: Pmax=? [F "goal"] [eps = 0.1]

method: AR, synthesis time: 0.03 s
number of holes: 11, family size: 1e7, quotient: 62 states / 296 actions
explored: 100 %
MDP stats: avg MDP size: 40, iterations: 22

feasible: yes
--------------------
 
INFO:2024-10-18 11:07:50,787 - (0.03s) - verify.py - Found trace: ['normal', 'normal', 'normal'] 
INFO:2024-10-18 11:07

In [18]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

# player_path = [(0, [])]
poss = product.simulate_paynt_assignment(assignment, 1000)
player_path = poss

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in product.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=[pos=0] !g0 accepting !s0 !l0 init

	--[0, 1:normal]-->
s3, labels=!g0 !l2 !s4 [pos=4] accepting normal
	--[1, 1:normal]-->
s9, labels=!g0 !l5 !s8 [pos=8] accepting normal
	--[2, 1:normal]-->
s14, labels=!g0 !l9 !s8 [pos=8] accepting normal
	--[3, 4:end]-->
s2, labels=stop
it took 11 tries until the goal was reached


<IPython.core.display.Javascript object>